# Kaggle Dataset Preparation: MIMIC-CXR AP/PA Subset

This notebook prepares a clean AP/PA frontal chest X-ray subset from the Kaggle MIMIC-CXR dataset and exports it for Colab embedding extraction.

In [ ]:
import ast
import os
import shutil
from pathlib import Path

import pandas as pd

In [ ]:
DATASET_ROOT = "/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset"
IMAGE_ROOT = os.path.join(DATASET_ROOT, "official_data_iccv_final")
TRAIN_CSV = os.path.join(DATASET_ROOT, "mimic_cxr_aug_train.csv")
VAL_CSV = os.path.join(DATASET_ROOT, "mimic_cxr_aug_validate.csv")

print("Dataset files:", os.listdir(DATASET_ROOT))
print("Image root exists:", Path(IMAGE_ROOT).exists())
print("Image root content:", os.listdir(IMAGE_ROOT)[:5])

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Columns:", train_df.columns.tolist())

display(train_df.head())

In [ ]:
def parse_list_cell(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

rows = []

for idx, row in train_df.iterrows():
    images = parse_list_cell(row["image"])
    views = parse_list_cell(row["view"])

    for i, img_rel in enumerate(images):
        view = views[i] if i < len(views) else None

        rows.append({
            "source_row": idx,
            "subject_id": row["subject_id"],
            "image_relative_path": img_rel,
            "image_path": os.path.join(IMAGE_ROOT, img_rel),
            "view": view,
            "text": row["text"],
            "text_augment": row["text_augment"],
            "AP": row["AP"],
            "PA": row["PA"],
            "Lateral": row["Lateral"],
        })

expanded_df = pd.DataFrame(rows)

print("Expanded shape:", expanded_df.shape)
print(expanded_df["view"].value_counts(dropna=False))
display(expanded_df.head())

In [ ]:
frontal_df = expanded_df[expanded_df["view"].isin(["AP", "PA"])].copy()

print("Frontal shape:", frontal_df.shape)
print(frontal_df["view"].value_counts())
display(frontal_df.head())

In [ ]:
def create_existing_subset(frontal_df, target_n, seed):
    shuffled = frontal_df.sample(frac=1, random_state=seed).reset_index(drop=True)

    valid_rows = []
    for _, row in shuffled.iterrows():
        if Path(row["image_path"]).exists():
            valid_rows.append(row)
        if len(valid_rows) >= target_n:
            break

    subset_df = pd.DataFrame(valid_rows).reset_index(drop=True)
    subset_df["image_id"] = [f"img_{i:06d}" for i in range(len(subset_df))]

    print("Subset shape:", subset_df.shape)
    print("Existing files:", subset_df["image_path"].apply(lambda p: Path(p).exists()).mean())
    print(subset_df["view"].value_counts())

    return subset_df

In [ ]:
subset_500_df = create_existing_subset(frontal_df, target_n=500, seed=42)
display(subset_500_df.head())

In [ ]:
subset_3000_df = create_existing_subset(frontal_df, target_n=3000, seed=123)
display(subset_3000_df.head())

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sample_path = subset_3000_df.loc[0, "image_path"]

print(sample_path)
print("Exists:", Path(sample_path).exists())

img = Image.open(sample_path)
print("Image size:", img.size)

plt.imshow(img, cmap="gray")
plt.axis("off")
plt.show()

In [ ]:
def export_subset_for_colab(subset_df, subset_size):
    image_dir = f"/kaggle/working/subset_{subset_size}_images"
    output_dir = f"/kaggle/working/outputs_{subset_size}"

    os.makedirs(image_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    for _, row in subset_df.iterrows():
        src = row["image_path"]
        dst = os.path.join(image_dir, row["image_id"] + ".jpg")
        shutil.copy2(src, dst)

    subset_export_df = subset_df.copy()
    subset_export_df["local_image_path"] = subset_export_df["image_id"] + ".jpg"

    subset_export_df.to_csv(
        os.path.join(output_dir, f"selected_metadata_{subset_size}_colab.csv"),
        index=False
    )

    subset_export_df[["image_id", "local_image_path", "view", "subject_id"]].to_csv(
        os.path.join(output_dir, f"image_ids_{subset_size}_colab.csv"),
        index=False
    )

    summary = f"""
# Dataset Summary - {subset_size} Image Subset

Dataset: Kaggle mimic-cxr-dataset

Original train rows: {train_df.shape[0]}
Expanded image rows: {expanded_df.shape[0]}
Frontal AP/PA image rows: {frontal_df.shape[0]}

Selected subset size: {subset_df.shape[0]}
Existing files: {subset_df["image_path"].apply(lambda p: Path(p).exists()).mean()}

View distribution:
{subset_df["view"].value_counts().to_string()}

Subset strategy:
- AP/PA frontal chest X-rays only
- existing image files only
- exported with local image filenames for Colab
"""

    with open(os.path.join(output_dir, f"dataset_summary_{subset_size}.md"), "w") as f:
        f.write(summary)

    print(f"Copied images: {len(os.listdir(image_dir))}")
    print(f"Saved metadata to: {output_dir}")
    print(summary)

In [ ]:
export_subset_for_colab(subset_3000_df, subset_size=3000)

In [ ]:
!cd /kaggle/working && zip -qr subset_3000_for_colab.zip subset_3000_images outputs_3000
!ls -lh /kaggle/working/subset_3000_for_colab.zip